In [0]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create or get the Spark session
spark = SparkSession.builder.appName("credit_risk").getOrCreate()

# Read the CSV from the Unity Catalog volume
df = spark.read.csv(
    "/Volumes/workspace/default/credit_risk_data/cs-training.csv",
    header=True,
    inferSchema=True
)

# Rename columns to match the dbt model names
df = df.withColumnRenamed("_c0", "id") \
       .withColumnRenamed("SeriousDlqin2yrs", "target_delinquency") \
       .withColumnRenamed("RevolvingUtilizationOfUnsecuredLines", "revolving_utilization") \
       .withColumnRenamed("NumberOfTime30-59DaysPastDueNotWorse", "days_30_59_late") \
       .withColumnRenamed("DebtRatio", "debt_ratio") \
       .withColumnRenamed("MonthlyIncome", "monthly_income") \
       .withColumnRenamed("NumberOfOpenCreditLinesAndLoans", "open_credit_lines") \
       .withColumnRenamed("NumberOfTimes90DaysLate", "times_90_late") \
       .withColumnRenamed("NumberRealEstateLoansOrLines", "real_estate_loans") \
       .withColumnRenamed("NumberOfTime60-89DaysPastDueNotWorse", "days_60_89_late") \
       .withColumnRenamed("NumberOfDependents", "dependents")

df.printSchema()
df.show(5)

root
 |-- id: integer (nullable = true)
 |-- target_delinquency: integer (nullable = true)
 |-- revolving_utilization: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- days_30_59_late: integer (nullable = true)
 |-- debt_ratio: double (nullable = true)
 |-- monthly_income: string (nullable = true)
 |-- open_credit_lines: integer (nullable = true)
 |-- times_90_late: integer (nullable = true)
 |-- real_estate_loans: integer (nullable = true)
 |-- days_60_89_late: integer (nullable = true)
 |-- dependents: string (nullable = true)

+---+------------------+---------------------+---+---------------+-----------+--------------+-----------------+-------------+-----------------+---------------+----------+
| id|target_delinquency|revolving_utilization|age|days_30_59_late| debt_ratio|monthly_income|open_credit_lines|times_90_late|real_estate_loans|days_60_89_late|dependents|
+---+------------------+---------------------+---+---------------+-----------+--------------+------------

In [0]:
# Import SQL functions for conditional logic
from pyspark.sql.functions import col, when, expr

# Safely cast monthly_income (contains 'NA' strings) to double
income_num = expr("try_cast(monthly_income as double)")

# Create income bands
df = df.withColumn(
    "income_band",
    when(income_num.isNull(), "Unknown")
    .when(income_num < 2000, "Very Low")
    .when(income_num < 5000, "Low")
    .when(income_num < 10000, "Medium")
    .when(income_num < 20000, "High")
    .otherwise("Very High")
)

# Create age bands
df = df.withColumn(
    "age_band",
    when(col("age") < 30, "18-29")
    .when(col("age") < 45, "30-44")
    .when(col("age") < 60, "45-59")
    .otherwise("60+")
)

df.select("age", "age_band", "monthly_income", "income_band").show(5)

+---+--------+--------------+-----------+
|age|age_band|monthly_income|income_band|
+---+--------+--------------+-----------+
| 45|   45-59|          9120|     Medium|
| 40|   30-44|          2600|        Low|
| 38|   30-44|          3042|        Low|
| 30|   30-44|          3300|        Low|
| 49|   45-59|         63588|  Very High|
+---+--------+--------------+-----------+
only showing top 5 rows


In [0]:
# Import aggregation functions
from pyspark.sql.functions import count, sum, round, col

# Aggregate delinquency rate by income and age bands
mart = df.groupBy("income_band", "age_band").agg(
    count("*").alias("total_customers"),
    sum("target_delinquency").alias("delinquent_customers")
)

# Calculate delinquency rate percentage
mart = mart.withColumn(
    "delinquency_rate_pct",
    round(col("delinquent_customers") / col("total_customers") * 100, 2)
)

# Sort by delinquency rate descending
mart = mart.orderBy(col("delinquency_rate_pct").desc())

mart.show(20)

+-----------+--------+---------------+--------------------+--------------------+
|income_band|age_band|total_customers|delinquent_customers|delinquency_rate_pct|
+-----------+--------+---------------+--------------------+--------------------+
|  Very High|   18-29|             16|                   3|               18.75|
|    Unknown|   18-29|           1442|                 211|               14.63|
|        Low|   18-29|           3661|                 458|               12.51|
|        Low|   30-44|          13233|                1552|               11.73|
|   Very Low|   18-29|           2628|                 280|               10.65|
|   Very Low|   30-44|           2643|                 277|               10.48|
|    Unknown|   30-44|           5413|                 556|               10.27|
|        Low|   45-59|          12685|                1218|                 9.6|
|   Very Low|   45-59|           2509|                 223|                8.89|
|  Very High|   30-44|      

## Validation: dbt vs. PySpark

The same transformation logic implemented in dbt (PostgreSQL) was reproduced in PySpark (Databricks). Both pipelines produce identical delinquency rates per segment, validating that the data model is consistent across the two environments.

This cross-validation is important because it proves that:

1. The business logic (income bands, age bands, delinquency rate) is correctly specified.
2. Different tools can implement the same model and arrive at the same result.
3. The pipeline is not tied to a single technology stack.

See the matching results in the `mart_risk_by_segment` table (dbt) and the `mart` DataFrame above (PySpark).